# AuscultTrust: Colab quickstart

Runs the code end to end on one short training unit: checks the GPU, installs the pinned training environment,
downloads ICBHI, checks the BEATs weights, runs the tests, trains the baseline (BEATs + PAFA) for `SMOKE_EPOCHS`
epochs, and exports frozen predictions. The full 100-epoch run is opt-in (`RUN_FULL`).

**Before you start**
- Runtime > Change runtime type > **GPU** (T4 or better). The notebook stops if no GPU is found.
- Code: the notebook uses `MyDrive/auscult-trust-delivery.zip` if present (see `DELIVERY.md`), else clones the public GitHub repo.
- ICBHI is downloaded by the notebook. BEATs weights are a manual download (see the check cell).
- Time: setup ~15 min; smoke run (2 epochs) about 10 min on a T4; full run several hours.
- Data is not redistributed with the code. You accept the ICBHI and KAUH terms when you use them.

In [ ]:
import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
ALLOW_CPU = os.environ.get("ALLOW_CPU") == "1"   # set only to test this notebook on a CPU machine
SMOKE_EPOCHS = int(os.environ.get("SMOKE_EPOCHS", "2"))
RUN_FULL = False                                  # True: 100-epoch baseline (hours)
RUN_ID = "colab-smoke"

if IN_COLAB:
    REPO = pathlib.Path("/content/auscult-trust")
    DRIVE = pathlib.Path("/content/drive/MyDrive")
    DATA_DIR = DRIVE / "auscult-trust-data"       # persists between sessions
else:
    here = pathlib.Path.cwd()
    REPO = here.parent if here.name == "notebooks" else here
    DRIVE = None
    DATA_DIR = pathlib.Path(os.environ.get("AT_DATA", REPO.parent / "data"))
os.environ["AT_DATA"] = str(DATA_DIR)
os.environ["PYTHONPATH"] = str(REPO)
print("repo:", REPO)
print("data:", DATA_DIR)

In [ ]:
import torch
print("torch", torch.__version__, "cuda build", torch.version.cuda, "available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
elif not ALLOW_CPU:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then restart the runtime and run again.")

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("data dir on Drive:", DATA_DIR)

In [ ]:
import zipfile
ZIP = (DRIVE / "auscult-trust-delivery.zip") if DRIVE else None
REPO_URL = "https://github.com/hiimmuc/KHKT2026-auscult-trust.git"
if IN_COLAB and not REPO.exists():
    if ZIP.exists():                      # delivery zip on Drive
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/content")
    else:                                 # no zip: clone the public repo
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
print("code present:", (REPO / "auscult_trust").is_dir())

In [ ]:
def sh(cmd, env=None):
    print("$", cmd, flush=True)
    full = {**os.environ, "MPLBACKEND": "Agg", **(env or {})}
    subprocess.run(cmd, shell=True, cwd=REPO, env=full, check=True)

PY = REPO / ".venv-train" / "bin" / "python"
if not PY.exists():
    sh("pip install -q uv")
    sh("uv venv .venv-train --python 3.10")
    sh("uv pip sync --python .venv-train/bin/python --index-strategy unsafe-best-match "
       "--extra-index-url https://download.pytorch.org/whl/cu128 requirements-train.lock")
sh(f"{PY} -c \"import torch, timm; print('env ok', torch.__version__, timm.__version__)\"")

In [ ]:
# Persist outputs on Drive so a session reset does not lose the run.
if DRIVE:
    out_drive = DRIVE / "auscult-trust-outputs"
    out_drive.mkdir(parents=True, exist_ok=True)
    link = REPO / "outputs"
    if not link.exists():
        link.symlink_to(out_drive, target_is_directory=True)
print("outputs ->", (REPO / "outputs").resolve())

In [ ]:
sh("bash scripts/download_data.sh --icbhi-only")
beats = DATA_DIR / "models" / "beats" / "BEATs_iter3_plus_AS2M.pt"
if not beats.exists():
    raise FileNotFoundError(
        f"Missing {beats}. Download BEATs_iter3_plus_AS2M.pt from the BEATs release "
        "(https://github.com/microsoft/unilm/tree/master/beats), create the folder and upload the file there, then rerun this cell."
    )
sh(f"{PY} scripts/prepare_data.py")

In [ ]:
sh(f"{PY} -m pytest -q tests")

In [ ]:
C = "experiments/train"
TRAIN = (f"{PY} -m auscult_trust.training.main "
         f"--config {C}/base.yaml,{C}/encoder_beats.yaml,{C}/grid/pafa_beats.yaml "
         "--cell beats+ft+pafa --selection fixed --seed 0 --num_workers 2 ")
sh(TRAIN + f"--epochs {SMOKE_EPOCHS} --report_epochs {SMOKE_EPOCHS}", env={"RUN_ID": RUN_ID})

In [ ]:
UNIT = REPO / "outputs" / "train" / RUN_ID / "beats+ft+pafa" / "seed0"
sh(f"{PY} -m auscult_trust.training.export_probs {UNIT} --ckpt report_epoch_{SMOKE_EPOCHS}.pth --num_workers 2")
import numpy as np, json
z = np.load(UNIT / "export" / "test.npz", allow_pickle=True)
print("test probs", z["probs"].shape if "probs" in z else z.files)
rep = json.loads((UNIT / "report.json").read_text())
for h in rep["history"][-1:]:
    print({k: h.get(k) for k in ("epoch", "train_loss", "test_score")})

In [ ]:
if RUN_FULL:
    sh(TRAIN + "--epochs 100 --report_epochs 100", env={"RUN_ID": "colab-full"})
else:
    print("RUN_FULL is False: skipped the 100-epoch run.")

## Outputs

- `outputs/train/colab-smoke/beats+ft+pafa/seed0/`: `report.json`, `test_preds.npz`, `train_args.json`, `export/test.npz`.
- `checkpoints/train/colab-smoke/...`: weights (`report_epoch_<E>.pth`).
- With Drive mounted, both live under `MyDrive/auscult-trust-outputs/` and `MyDrive/auscult-trust-data/` survives resets.